# March Mania · Ranking resolution and cardinal strength
**Round 14 · four candidates · two families · fixed consensus reference**

Two independently preregistered rounds, not one large search. The reference is now the completed **17-input model including consensus**. This notebook does not load round 15 results or adopt its selected features. No automatic follow-on job runs.

**Scope:** men's 2022–2025 played main draw, training on earlier seasons. These years were already used in the project, so this is exploratory rather than untouched testing. No 2026 outcomes or submission are used. It is not the production pooled XGBoost model.

Run the terminal tests in **START_HERE.md**. Use **Python (March Mania)**. No installation, cloud calls, Git writes, archive downloads, rating fits, or raw-data modifications.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink
KIT = Path.cwd().resolve()
if not (KIT / "run_round.py").is_file():
    KIT = Path.home() / "march_feature_rounds_14_15"
assert (KIT / "run_round.py").is_file(), "Open this notebook inside the extracted kit."
sys.path.insert(0, str(KIT))
from run_round import run_stage
from round_plots import figures
ROUND = "14"
REPORTS = KIT / "reports" / ("round" + ROUND)
pio.renderers.default = "plotly_mimetype"
print("Kernel:", sys.executable)
print("Round:", ROUND, "| kit:", KIT)
print("Rating fits: 0 | new classifier cap: 16 | reference replays: 4")

## 1. Preserve the actual milestone 13 result
Consensus lowered mean later-era Brier by 0.0006879 and improved three of four seasons. Most of that gain came from 2024. The gate warranted a **separate production-recipe test**, not automatic promotion. That production transfer remains pending; these two requested new feature rounds do not substitute for it.

The following table is the prior returned result, not a newly measured result.

In [ ]:
prior = pd.read_csv(KIT / "evidence/round13/metrics.csv")
display(prior[["Season","recipe","brier","log_loss","delta_vs_anchor"]].round(7))
print(json.dumps(json.loads((KIT / "evidence/round13/decisions.json").read_text()), indent=2))

## 2. Build four candidates without refitting ratings
**Family A — tail resolution (2).** Unclipped midpoint logit and variance-scaled normal-quantile corrections preserve ordinal separation that the reference's 1% clip can hide. These are fixed representation hypotheses, not estimated win probabilities.

**Family B — cardinal mapping (2).** Map published consensus positions onto the contemporaneous league distributions of margin strength and adjusted net efficiency. This assumes those distributions help interpret ordinal spacing; it does not recover any ranking vendor's private numerical ratings.

Use only cached latest legal publication panels. No new ranking model is fitted. Mapping distributions are season-local pre-tournament inputs, not label-fitted transforms.

**Preparation ceiling: 300 seconds. Heartbeats: 15 seconds.** Reuse 12 base snapshots and four reference models. Potential seeded pairings are built before tournament outcomes are attached.

In [ ]:
run_stage(ROUND, "prepare", max_seconds=300)
RUN = Path(json.loads((REPORTS / "latest_run.json").read_text())["run_dir"])
print(json.dumps(json.loads((RUN / "prepare.json").read_text()), indent=2))
display(pd.read_csv(RUN / "feature_registry.csv").query("new_candidate == True"))
display(pd.read_csv(RUN / "coverage.csv"))
display(pd.read_csv(RUN / "prior_replay.csv").round(7))

## 3. Controlled evaluation, including a duplication diagnostic
| Configuration | Inputs | Purpose |
|---|---:|---|
| Reference | 17 | Verified existing consensus model |
| Reference + Family A | 19 | Add-alone family comparison |
| Reference + Family B | 19 | Add-alone family comparison |
| Reference + both | 21 | Declared primary comparison |
| Reference + four exact copied inputs | 21 | Regularization-sensitivity diagnostic; no new information |

Four later validation seasons give 20 comparisons: four replays and **at most 16 new classifier fits**. Logistic C=0.1, mirrored orientations, per-game weighting and training-only RMS scaling remain fixed. Only training-constant inputs may be removed. Duplicates can change the effective penalty; this diagnostic helps prevent calling such an effect new predictive information.

**Evaluation ceiling: 180 seconds.** Negative `delta_vs_anchor` is better. Primary: `both_given_reference`. Secondary add/drop contrasts keep all other inputs fixed.

In [ ]:
run_stage(ROUND, "evaluate", max_seconds=180)
metrics = pd.read_csv(RUN / "metrics.csv")
display(metrics[["Season","recipe","brier","log_loss","delta_vs_anchor","active_features","source"]].round(7))
display(pd.read_csv(RUN / "ablations.csv").round(7))
print(json.dumps(json.loads((RUN / "decisions.json").read_text()), indent=2))
print(json.dumps(json.loads((RUN / "evaluation_receipt.json").read_text()), indent=2))

## 4. Save the scientific report before rendering charts
Expansion requires mean primary Brier change ≤ −0.0005, at least three of four seasons improving, worst deterioration ≤ +0.003, and a negative mean difference versus the duplicate-input control. These are resource-allocation gates, not significance tests. Neither outcome modifies the other round's protocol.

**Reporting ceiling: 120 seconds.** The return ZIP excludes raw rows, fitted models, per-game predictions, and private edited notebooks. A scientific `DO_NOT_PROMOTE` still exports normally.

In [ ]:
run_stage(ROUND, "report", max_seconds=120)
record = json.loads((REPORTS / "latest_report.json").read_text())
print("Scientific archive:", record["return_zip"])
print("Interactive HTML:", record["html"])
display(FileLink(str(Path(record["return_zip"]).relative_to(KIT))))
display(FileLink(str(Path(record["html"]).relative_to(KIT))))

## 5. Ten interactive Plotly charts
Read season stability, support, primary and controlled family effects, calibration, overlap and coefficients together. Correlation is diagnostic, not a proof of causality. The HTML version is already saved.

In [ ]:
plots = figures(RUN, ROUND, KIT / "evidence/round13")
assert len(plots) == 10
for fig in plots[:5]:
    fig.show()

In [ ]:
for fig in plots[5:]:
    fig.show()

## 6. Checkpoint and next bounded action
Save this notebook with Ctrl+S. Keep every private_runs folder. After round 14 completes technically, shut down this kernel and open round 15. A scientific failure does not change or prevent the independent second experiment. A technical failure stops for diagnosis.


In [ ]:
print("Completed artifacts:", RUN)
if ROUND == "15" and (KIT / "reports/round14/milestone_14_return.zip").is_file():
    from package_returns import package
    combined = package(KIT)
    print("Attach this combined return:", combined)
    display(FileLink(str(combined.relative_to(KIT))))
else:
    print("Return file:", record["return_zip"])
print("No submission or GitHub change has been made. Save the executed notebook.")